# Time series analysis at the aggregate level

In [1]:
import pandas as pd
import numpy as np
import random 
from grid_search import estimate_single_config

from joblib import Parallel, delayed
from tqdm.auto import tqdm

import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm

import plot_style

plot_style.apply()
random.seed(42)

In [2]:
# load feature matrix and response variable
feature_matrix = pd.read_csv("../../data/merged_return_topic_data.csv", index_col=0, parse_dates=True)
response_variables = pd.read_csv("../../data/response.csv", index_col=0, parse_dates=True)

In [3]:
def to_ar1_innovations(X: pd.DataFrame, min_obs: int = 30) -> pd.DataFrame:
    """Return AR(1) innovations (residuals) for each column of X."""
    X_innov = pd.DataFrame(index=X.index, columns=X.columns, dtype="float64")

    for col in X.columns:
        s = pd.to_numeric(X[col], errors="coerce")
        tmp = pd.DataFrame({"x": s, "x_lag1": s.shift(1)}).dropna()
        if len(tmp) < min_obs or tmp["x"].nunique() < 3 or tmp["x_lag1"].nunique() < 3:
            continue

        res = sm.OLS(tmp["x"], sm.add_constant(tmp["x_lag1"])).fit()
        X_innov.loc[tmp.index, col] = res.resid

    return X_innov

In [4]:
# set seed
random.seed(42)

# load feature matrix and response variable
feature_matrix = pd.read_csv("../../data/merged_return_topic_data.csv", index_col=0, parse_dates=True)
response_variables = pd.read_csv("../../data/response.csv", index_col=0, parse_dates=True)

# select a response variable (either marekt return or sp500 return)
y = response_variables['vwretx']  # or 'sprtrn' for SP500 returns or vwretx

# # transform returns to log
y = np.log(y+1)

# create feature matrix
X = feature_matrix.copy()

# Separate topic (name) and stock (numeric) columns
topic_cols = [col for col in X.columns if not str(col).isdigit()]
stock_cols = [col for col in X.columns if str(col).isdigit()]

# Final filtered dataframe
X = X[topic_cols]

# # usage
X = to_ar1_innovations(X)

# remove the first row wiht iloc
X = X.iloc[1:]

# ensure that all indices align
common_index = X.index.intersection(y.index)
X = X.loc[common_index]
y = y.loc[common_index]

In [5]:
#import the best hyperparameters from the file
best_hyperparameters = {}
with open("best_hyperparameters.txt", "r") as f:
    for line in f:
        key, value = line.strip().split(": ")
        best_hyperparameters[key] = float(value)

#define the paramters for the esitmation
window_size = int(best_hyperparameters.get('window_size'))
n_lags = int(best_hyperparameters.get('n_lags'))
lambda_val = best_hyperparameters.get('lambda')
print(f"Using hyperparameters: window_size={window_size}, n_lags={n_lags}, lambda={lambda_val}")

Using hyperparameters: window_size=360, n_lags=21, lambda=0.002536652371477602


In [6]:
#run a single configuration estimation
results_df = estimate_single_config(X, y, window_size, n_lags, lambda_val)

In [11]:
results_df['details']

,date,window_size,n_lags,lambda,window_index,lasso_intercept,lasso_r2_in,num_nonzero_coefficients,Lasso_Natural disasters_lag_1,Lasso_Internet_lag_1,...,Lasso_European politics_lag_21,Lasso_Size_lag_21,Lasso_NASD_lag_21,Lasso_Mexico_lag_21,Lasso_Retail_lag_21,Lasso_Long/short term_lag_21,Lasso_Wide range_lag_21,Lasso_Lawsuits_lag_21,Lasso_UK_lag_21,Lasso_Revenue growth_lag_21
0,2011-07-08,360,21,0.002537,0,0.0,-0.000605,0.0,-0.0,-0.0,...,-0.0,0.0,0.0,-0.0,0.0,-0.0,-0.0,-0.0,-0.0,-0.0
1,2011-07-11,360,21,0.002537,1,0.0,-0.000797,0.0,-0.0,-0.0,...,-0.0,0.0,0.0,-0.0,0.0,-0.0,-0.0,-0.0,0.0,-0.0
2,2011-07-12,360,21,0.002537,2,0.0,-0.000704,0.0,-0.0,-0.0,...,-0.0,0.0,0.0,-0.0,0.0,-0.0,-0.0,-0.0,0.0,-0.0
3,2011-07-13,360,21,0.002537,3,0.0,-0.000905,0.0,-0.0,-0.0,...,-0.0,0.0,0.0,-0.0,0.0,-0.0,-0.0,-0.0,0.0,-0.0
4,2011-07-14,360,21,0.002537,4,0.0,-0.000592,0.0,-0.0,-0.0,...,-0.0,0.0,0.0,-0.0,0.0,-0.0,-0.0,-0.0,0.0,-0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1503,2017-06-28,360,21,0.002537,1503,0.0,-0.003320,0.0,-0.0,-0.0,...,0.0,0.0,-0.0,0.0,-0.0,-0.0,-0.0,-0.0,0.0,-0.0
1504,2017-06-28,360,21,0.002537,1504,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1505,2017-06-29,360,21,0.002537,1505,0.0,-0.002325,0.0,-0.0,-0.0,...,0.0,0.0,-0.0,0.0,-0.0,-0.0,-0.0,-0.0,0.0,-0.0
1506,2017-06-29,360,21,0.002537,1506,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
